# Twitter sentiment and stock prices

Relationship between daily Twitter sentiment and stock prices for Tesla (TSLA) and Apple (AAPL).

Sentiment comes from `cardiffnlp/twitter-roberta-base-sentiment`. The compound score for a tweet is the positive-class probability minus the negative-class probability. A day's sentiment is the mean compound score of that day's tweets.

Put `stock_tweets.csv` and `stock_yfinance_data.csv` in the working directory before running the data cells. The model is public. If you switch to a gated model, set `HF_TOKEN` in the environment. Do not paste tokens into this notebook.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from sklearn.metrics import mean_absolute_error, mean_squared_error
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.stattools import grangercausalitytests
from transformers import AutoModelForSequenceClassification, AutoTokenizer

TWEETS_PATH = Path("stock_tweets.csv")
PRICES_PATH = Path("stock_yfinance_data.csv")
MODEL_NAME = "cardiffnlp/twitter-roberta-base-sentiment"

# Window used for the price-versus-sentiment figures.
PLOT_START, PLOT_END = "2021-09-01", "2022-09-01"
# Window used for the Granger tests.
TEST_START, TEST_END = "2021-11-01", "2022-12-31"


## Load prices and tweets

Dates are floored to the calendar day and stored as timezone-naive timestamps so tweet and price rows line up.

In [ ]:
def to_day(series):
    """Parse timestamps and floor them to a timezone-naive calendar day."""
    parsed = pd.to_datetime(series, utc=True)
    return parsed.dt.tz_convert(None).dt.floor("D")


def load_frames(tweets_path=TWEETS_PATH, prices_path=PRICES_PATH):
    if not tweets_path.exists() or not prices_path.exists():
        missing = [str(path) for path in (tweets_path, prices_path) if not path.exists()]
        raise FileNotFoundError(
            "Missing data file(s): " + ", ".join(missing) + ". "
            "Place them in the working directory and rerun."
        )

    tweets = pd.read_csv(tweets_path)
    prices = pd.read_csv(prices_path)
    tweets["Date"] = to_day(tweets["Date"])
    prices["Date"] = to_day(prices["Date"])
    tweets["Tweet"] = tweets["Tweet"].fillna("").astype(str)
    return tweets, prices


tweets, prices = load_frames()
print(f"Tweets: {len(tweets):,}  Prices: {len(prices):,}")
print("Symbols:", ", ".join(sorted(tweets["Stock Name"].dropna().unique())))


## Score tweets

Labels from this checkpoint are negative (`0`), neutral (`1`), and positive (`2`).

In [ ]:
def compound_scores(texts, tokenizer, model, batch_size=32):
    """Positive probability minus negative probability for each text."""
    scores = []
    model.eval()
    for start in range(0, len(texts), batch_size):
        batch = list(texts[start : start + batch_size])
        inputs = tokenizer(
            batch,
            return_tensors="pt",
            truncation=True,
            padding=True,
            max_length=512,
        )
        with torch.no_grad():
            logits = model(**inputs).logits
        probs = torch.softmax(logits, dim=1)
        scores.extend((probs[:, 2] - probs[:, 0]).tolist())
    return scores


tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME)

tweets = tweets.copy()
tweets["RoBERTa_Sentiment"] = compound_scores(tweets["Tweet"], tokenizer, model)


## Daily panels

Each symbol gets one row per trading day: close price, mean sentiment, and the number of tweets that day.

In [ ]:
def daily_panel(tweets, prices, symbol, start=None, end=None):
    symbol_tweets = tweets.loc[tweets["Stock Name"] == symbol]
    daily_sentiment = symbol_tweets.groupby("Date")["RoBERTa_Sentiment"].agg(
        RoBERTa_Sentiment="mean",
        n_tweets="size",
    )

    symbol_prices = prices.loc[prices["Stock Name"] == symbol].copy()
    symbol_prices = symbol_prices.drop_duplicates("Date", keep="last").set_index("Date")

    panel = symbol_prices[["Close"]].join(daily_sentiment, how="inner").dropna()
    panel = panel.sort_index()
    if start is not None or end is not None:
        panel = panel.loc[start:end]
    return panel


def add_normalized_columns(panel):
    panel = panel.copy()
    close_span = panel["Close"].max() - panel["Close"].min()
    panel["Close_norm"] = (
        0.0 if close_span == 0 else (panel["Close"] - panel["Close"].min()) / close_span
    )
    sent_min = panel["RoBERTa_Sentiment"].min()
    sent_span = panel["RoBERTa_Sentiment"].max() - sent_min
    panel["Sentiment_norm"] = (
        0.0 if sent_span == 0 else (panel["RoBERTa_Sentiment"] - sent_min) / sent_span
    )
    return panel


tesla = add_normalized_columns(daily_panel(tweets, prices, "TSLA", PLOT_START, PLOT_END))
apple = add_normalized_columns(daily_panel(tweets, prices, "AAPL", PLOT_START, PLOT_END))
print("Tesla days:", len(tesla), " Apple days:", len(apple))
tesla.head()


## Price and sentiment

Min-max normalized close and sentiment over September 2021 through September 2022.

In [ ]:
def plot_price_vs_sentiment(panel, title):
    fig, ax = plt.subplots(figsize=(14, 6))
    ax.plot(panel.index, panel["Close_norm"], label="Normalized close", color="tab:blue")
    ax.plot(
        panel.index,
        panel["Sentiment_norm"],
        label="Normalized sentiment",
        color="tab:orange",
    )
    ax.set_title(title)
    ax.set_xlabel("Date")
    ax.set_ylabel("Normalized value")
    ax.legend()
    ax.grid(True, alpha=0.3)
    fig.autofmt_xdate()
    fig.tight_layout()
    return fig


plot_price_vs_sentiment(
    tesla,
    "Tesla close vs. Twitter sentiment (normalized), Sep 2021 – Sep 2022",
)
plt.show()


## Lagged sentiment

Same Tesla window. Sentiment is lagged by one and two trading days.

In [ ]:
lagged = tesla.copy()
lagged["Sentiment_lag1"] = lagged["RoBERTa_Sentiment"].shift(1)
lagged["Sentiment_lag2"] = lagged["RoBERTa_Sentiment"].shift(2)

fig, ax_price = plt.subplots(figsize=(14, 6))
ax_price.plot(lagged.index, lagged["Close"], label="Tesla close ($)", color="tab:blue", linewidth=2)
ax_sent = ax_price.twinx()
ax_sent.plot(
    lagged.index,
    lagged["Sentiment_lag1"],
    label="Sentiment (lag = 1 day)",
    color="black",
    alpha=0.7,
)
ax_sent.plot(
    lagged.index,
    lagged["Sentiment_lag2"],
    label="Sentiment (lag = 2 days)",
    color="tab:green",
    alpha=0.7,
)
ax_price.set_title("Tesla close vs. lagged Twitter sentiment\nSep 2021 – Sep 2022")
ax_price.set_xlabel("Date")
ax_price.set_ylabel("Close price ($)")
ax_sent.set_ylabel("Sentiment score")

handles, labels = ax_price.get_legend_handles_labels()
handles2, labels2 = ax_sent.get_legend_handles_labels()
ax_sent.legend(handles + handles2, labels + labels2, loc="upper right")
fig.autofmt_xdate()
fig.tight_layout()
plt.show()


## Granger causality

Tests whether lagged sentiment helps predict close price, up to two trading days, from November 2021 through December 2022. A small p-value on the sentiment lags is evidence of that predictive relationship.

In [ ]:
def granger_sentiment_to_price(symbol, maxlag=2):
    panel = daily_panel(tweets, prices, symbol, TEST_START, TEST_END)
    print(f"=== {symbol}: sentiment -> close ({len(panel)} days) ===")
    grangercausalitytests(panel[["Close", "RoBERTa_Sentiment"]], maxlag=maxlag, verbose=True)


granger_sentiment_to_price("TSLA")
granger_sentiment_to_price("AAPL")


## ARIMA with extra regressors

ARIMA(1, 1, 0) on normalized Tesla close. Extra regressors are that day's sentiment, the weekday, and a 5-day moving average of normalized close shifted by one day so the average does not include the day being predicted.

In [ ]:
arima_frame = tesla.copy()
arima_frame["Day_of_Week"] = arima_frame.index.dayofweek
arima_frame["MA_5"] = arima_frame["Close_norm"].rolling(window=5).mean().shift(1)
arima_frame = arima_frame.dropna()

exog = arima_frame[["RoBERTa_Sentiment", "Day_of_Week", "MA_5"]]
arima = ARIMA(arima_frame["Close_norm"], order=(1, 1, 0), exog=exog)
arima_fit = arima.fit()
print(arima_fit.summary())


## Holdout forecast

ARIMA(2, 1, 0) fit on all but the last 10 Tesla days, then scored with mean absolute deviation, mean squared error, and mean absolute percentage error.

In [ ]:
series = tesla["Close_norm"]
train, test = series.iloc[:-10], series.iloc[-10:]

forecast_fit = ARIMA(train, order=(2, 1, 0)).fit()
forecast_values = forecast_fit.forecast(steps=len(test))

forecast = pd.DataFrame(
    {"Actual": test.to_numpy(), "Forecast": forecast_values.to_numpy()},
    index=test.index,
)
forecast["Absolute_Error"] = (forecast["Actual"] - forecast["Forecast"]).abs()

mad = mean_absolute_error(forecast["Actual"], forecast["Forecast"])
mse = mean_squared_error(forecast["Actual"], forecast["Forecast"])
mape = np.mean(np.abs((forecast["Actual"] - forecast["Forecast"]) / forecast["Actual"])) * 100

print(forecast)
print(f"MAD:  {mad:.4f}")
print(f"MSE:  {mse:.4f}")
print(f"MAPE: {mape:.2f}%")


## Example classifications

Spot-check the label mapping on a few sentences. Each line is the winning class, the compound score, and the class probabilities.

In [ ]:
LABELS = ["negative", "neutral", "positive"]
EXAMPLES = [
    "I love Tesla, it's the best car ever!",
    "I hate Tesla, it's terrible!",
    "Tesla is just okay, nothing special.",
    "Wow, Tesla has changed the game!",
    (
        "IDK who @Tesla should hear this, but @elonmusk can we please get a fix "
        "for basic summon at least for the MS Plaid? My car NEVER goes forward or "
        "backwards anymore just starts, then shuts off."
    ),
]

example_scores = compound_scores(EXAMPLES, tokenizer, model, batch_size=len(EXAMPLES))
inputs = tokenizer(
    EXAMPLES,
    return_tensors="pt",
    truncation=True,
    padding=True,
    max_length=512,
)
with torch.no_grad():
    probs = torch.softmax(model(**inputs).logits, dim=1).tolist()

for text, prob, compound in zip(EXAMPLES, probs, example_scores):
    label = LABELS[int(np.argmax(prob))]
    breakdown = ", ".join(f"{name}={value:.3f}" for name, value in zip(LABELS, prob))
    print(f"{label:8}  compound={compound:+.3f}  ({breakdown})")
    print(f"  {text}\n")
